In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from __future__ import print_function
import argparse

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import DataLoader
from model import Net as DBPNLL
from data import get_eval_set
from functools import reduce
from skimage.transform import pyramid_reduce, pyramid_expand

import scipy.io as sio
import time

In [3]:
for filename in os.listdir('/data/andria/TROPOMI_array_normalized/'):
    original_array = np.load('/data/andria/TROPOMI_array_normalized/'+filename)
    bic_downscaled = pyramid_reduce(original_array, downscale=16, sigma=None, order=3, mode='constant', cval=0)
    np.save('Input/Set5_LR_x16/down_'+filename, bic_downscaled)

In [4]:
model_w_16 = 'weights/MOD_tensorese-hivemindDBPNLL1channel_16_MSE.pth'
model_w_8 = 'weights/MOD_tensorese-hivemindDBPNLL1channel_8_MSE.pth'
model_w_4 = 'weights/MOD_tensorese-hivemindDBPNLL1channel_4_MSE.pth'
model_w_2 = 'weights/MOD_tensorese-hivemindDBPNLL1channel_2_MSE.pth'

gpus_list=range(1)
cuda = True
if cuda and not torch.cuda.is_available():
    raise Exception("No GPU found, please run without --cuda")

torch.manual_seed(123)
if cuda:
    torch.cuda.manual_seed(123)

print('===> Loading datasets')
test_set = get_eval_set(os.path.join('Input','Set5_LR_x16'), 16)
testing_data_loader = DataLoader(dataset=test_set, num_workers=1, batch_size=1, shuffle=False)

print('===> Building models')
model_16 = DBPNLL(num_channels=1, base_filter=64,  feat = 256, num_stages=10, scale_factor=16)
model_8 = DBPNLL(num_channels=1, base_filter=64,  feat = 256, num_stages=10, scale_factor=8)
model_4 = DBPNLL(num_channels=1, base_filter=64,  feat = 256, num_stages=10, scale_factor=4)
model_2 = DBPNLL(num_channels=1, base_filter=64,  feat = 256, num_stages=10, scale_factor=2)

if cuda:
    model_16 = torch.nn.DataParallel(model_16, device_ids=gpus_list)
    model_8 = torch.nn.DataParallel(model_8, device_ids=gpus_list)
    model_4 = torch.nn.DataParallel(model_4, device_ids=gpus_list)
    model_2 = torch.nn.DataParallel(model_2, device_ids=gpus_list)

model_16.load_state_dict(torch.load(model_w_16, map_location=lambda storage, loc: storage))
model_8.load_state_dict(torch.load(model_w_8, map_location=lambda storage, loc: storage))
model_4.load_state_dict(torch.load(model_w_4, map_location=lambda storage, loc: storage))
model_2.load_state_dict(torch.load(model_w_2, map_location=lambda storage, loc: storage))
print('Pre-trained SR model is loaded.')

if cuda:
    model_16 = model_16.cuda(gpus_list[0])
    model_8 = model_8.cuda(gpus_list[0])
    model_4 = model_4.cuda(gpus_list[0])
    model_2 = model_2.cuda(gpus_list[0])

def eval():
    model_16.eval()
    model_8.eval()
    model_4.eval()
    model_2.eval()
    for batch in testing_data_loader:
        with torch.no_grad():
            input, name = Variable(batch[0]), batch[1]
        if cuda:
            input = input.cuda(gpus_list[0])

        t0 = time.time()
        with torch.no_grad():
            prediction_16 = model_16(input)
        t1 = time.time()
        name_16 = name[0].replace('.npy', '_16.npy')
        print("===> Processing: %s || Path: 1-16 || Timer: %.4f sec." % (name_16, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_8 = model_8(input)
            prediction_8_2 = model_2(prediction_8)
        t1 = time.time()
        name_8_2 = name[0].replace('.npy', '_8_2.npy')
        print("===> Processing: %s || Path: 1-8-16 || Timer: %.4f sec." % (name_8_2, (t1 - t0)))
        
        
        t0 = time.time()
        with torch.no_grad():
            prediction_4 = model_4(input)
            prediction_4_4 = model_4(prediction_4)
        t1 = time.time()
        name_4_4 = name[0].replace('.npy', '_4_4.npy')
        print("===> Processing: %s || Path: 1-4-16 || Timer: %.4f sec." % (name_4_4, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_2 = model_2(input)
            prediction_2_4 = model_4(prediction_2)
            prediction_2_4_2 = model_2(prediction_2_4)
        t1 = time.time()
        name_2_4_2 = name[0].replace('.npy', '_2_4_2.npy')
        print("===> Processing: %s || Path: 1-2-8-16 || Timer: %.4f sec." % (name_2_4_2, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_4 = model_4(input)
            prediction_4_2 = model_2(prediction_4)
            prediction_4_2_2 = model_2(prediction_4_2)
        t1 = time.time()
        name_4_2_2 = name[0].replace('.npy', '_4_2_2.npy')
        print("===> Processing: %s || Path: 1-4-8-16 || Timer: %.4f sec." % (name_4_2_2, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_2 = model_2(input)
            prediction_2_2 = model_2(prediction_2)
            prediction_2_2_4 = model_4(prediction_2_2)
        t1 = time.time()
        name_2_2_4 = name[0].replace('.npy', '_2_2_4.npy')
        print("===> Processing: %s || Path: 1-2-4-16 || Timer: %.4f sec." % (name_2_2_4, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_2 = model_2(input)
            prediction_2_2 = model_2(prediction_2)
            prediction_2_2_2 = model_2(prediction_2_2)
            prediction_2_2_2_2 = model_2(prediction_2_2_2)
        t1 = time.time()
        name_2_2_2_2 = name[0].replace('.npy', '_2_2_2_2.npy')
        print("===> Processing: %s || Path: 1-2-4-8-16 || Timer: %.4f sec." % (name_2_2_2_2, (t1 - t0)))
        
        t0 = time.time()
        with torch.no_grad():
            prediction_2 = model_2(input)
            prediction_2_8 = model_8(prediction_2)
        t1 = time.time()
        name_2_8 = name[0].replace('.npy', '_2_8.npy')
        print("===> Processing: %s || Path: 1-2-16 || Timer: %.4f sec." % (name_2_8, (t1 - t0)))
        

        save_img(prediction_16.cpu().data, name_16)
        save_img(prediction_8_2.cpu().data, name_8_2)
        save_img(prediction_4_4.cpu().data, name_4_4)
        save_img(prediction_2_4_2.cpu().data, name_2_4_2)
        save_img(prediction_4_2_2.cpu().data, name_4_2_2)
        save_img(prediction_2_2_4.cpu().data, name_2_2_4)
        save_img(prediction_2_2_2_2.cpu().data, name_2_2_2_2)
        save_img(prediction_2_8.cpu().data, name_2_8)
        
        
def save_img(img, img_name):
    save_arr = img.numpy().squeeze(axis = 0).squeeze(axis = 0)
    # save img
    save_dir=os.path.join('Results/','Set5_LR_x16')
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        
    save_fn = save_dir +'/'+ img_name
    np.save(save_fn, save_arr)
    
eval()

===> Loading datasets
===> Building models
Pre-trained SR model is loaded.
===> Processing: down_S5P_NRTI_L2__O3_____20230307T090806_20230307T091306_27963_03_020401_20230307T094829_16.npy || Path: 1-16 || Timer: 1.1310 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230307T090806_20230307T091306_27963_03_020401_20230307T094829_8_2.npy || Path: 1-8-16 || Timer: 0.2874 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230307T090806_20230307T091306_27963_03_020401_20230307T094829_4_4.npy || Path: 1-4-16 || Timer: 0.0118 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230307T090806_20230307T091306_27963_03_020401_20230307T094829_2_4_2.npy || Path: 1-2-8-16 || Timer: 0.3140 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230307T090806_20230307T091306_27963_03_020401_20230307T094829_4_2_2.npy || Path: 1-4-8-16 || Timer: 0.2732 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230307T090806_20230307T091306_27963_03_020401_20230307T094829_2_2_4.npy || Path: 1-2-4-16 || Timer: 0.0381 sec.
===>

===> Processing: down_S5P_NRTI_L2__O3_____20230303T170306_20230303T170806_27911_03_020401_20230303T174857_2_2_2_2.npy || Path: 1-2-4-8-16 || Timer: 0.2948 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T170306_20230303T170806_27911_03_020401_20230303T174857_2_8.npy || Path: 1-2-16 || Timer: 0.0108 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230304T201806_20230304T202306_27927_03_020401_20230304T205741_16.npy || Path: 1-16 || Timer: 0.0059 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230304T201806_20230304T202306_27927_03_020401_20230304T205741_8_2.npy || Path: 1-8-16 || Timer: 0.2874 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230304T201806_20230304T202306_27927_03_020401_20230304T205741_4_4.npy || Path: 1-4-16 || Timer: 0.0108 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230304T201806_20230304T202306_27927_03_020401_20230304T205741_2_4_2.npy || Path: 1-2-8-16 || Timer: 0.3160 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230304T201806_20230304T202306_27927

===> Processing: down_S5P_NRTI_L2__O3_____20230304T202306_20230304T202806_27927_03_020401_20230304T205901_4_2_2.npy || Path: 1-4-8-16 || Timer: 0.2732 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230304T202306_20230304T202806_27927_03_020401_20230304T205901_2_2_4.npy || Path: 1-2-4-16 || Timer: 0.0381 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230304T202306_20230304T202806_27927_03_020401_20230304T205901_2_2_2_2.npy || Path: 1-2-4-8-16 || Timer: 0.2943 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230304T202306_20230304T202806_27927_03_020401_20230304T205901_2_8.npy || Path: 1-2-16 || Timer: 0.0109 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T103806_20230303T104306_27907_03_020401_20230303T124048_16.npy || Path: 1-16 || Timer: 0.0059 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T103806_20230303T104306_27907_03_020401_20230303T124048_8_2.npy || Path: 1-8-16 || Timer: 0.2865 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T103806_20230303T104306_2

===> Processing: down_S5P_NRTI_L2__O3_____20230303T120306_20230303T120806_27908_03_020401_20230303T125535_2_4_2.npy || Path: 1-2-8-16 || Timer: 0.3159 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T120306_20230303T120806_27908_03_020401_20230303T125535_4_2_2.npy || Path: 1-4-8-16 || Timer: 0.2736 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T120306_20230303T120806_27908_03_020401_20230303T125535_2_2_4.npy || Path: 1-2-4-16 || Timer: 0.0382 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T120306_20230303T120806_27908_03_020401_20230303T125535_2_2_2_2.npy || Path: 1-2-4-8-16 || Timer: 0.2948 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T120306_20230303T120806_27908_03_020401_20230303T125535_2_8.npy || Path: 1-2-16 || Timer: 0.0107 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T170806_20230303T171306_27911_03_020401_20230303T175028_16.npy || Path: 1-16 || Timer: 0.0064 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T170806_20230303T1713

===> Processing: down_S5P_NRTI_L2__O3_____20230303T120806_20230303T121306_27908_03_020401_20230303T125807_8_2.npy || Path: 1-8-16 || Timer: 0.2877 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T120806_20230303T121306_27908_03_020401_20230303T125807_4_4.npy || Path: 1-4-16 || Timer: 0.0109 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T120806_20230303T121306_27908_03_020401_20230303T125807_2_4_2.npy || Path: 1-2-8-16 || Timer: 0.3165 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T120806_20230303T121306_27908_03_020401_20230303T125807_4_2_2.npy || Path: 1-4-8-16 || Timer: 0.2743 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T120806_20230303T121306_27908_03_020401_20230303T125807_2_2_4.npy || Path: 1-2-4-16 || Timer: 0.0383 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T120806_20230303T121306_27908_03_020401_20230303T125807_2_2_2_2.npy || Path: 1-2-4-8-16 || Timer: 0.2954 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T120806_20230303T1

===> Processing: down_S5P_NRTI_L2__O3_____20230303T102806_20230303T103306_27907_03_020401_20230303T110419_16.npy || Path: 1-16 || Timer: 0.0059 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T102806_20230303T103306_27907_03_020401_20230303T110419_8_2.npy || Path: 1-8-16 || Timer: 0.2881 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T102806_20230303T103306_27907_03_020401_20230303T110419_4_4.npy || Path: 1-4-16 || Timer: 0.0108 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T102806_20230303T103306_27907_03_020401_20230303T110419_2_4_2.npy || Path: 1-2-8-16 || Timer: 0.3169 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T102806_20230303T103306_27907_03_020401_20230303T110419_4_2_2.npy || Path: 1-4-8-16 || Timer: 0.2746 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T102806_20230303T103306_27907_03_020401_20230303T110419_2_2_4.npy || Path: 1-2-4-16 || Timer: 0.0383 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T102806_20230303T103306_27907

===> Processing: down_S5P_NRTI_L2__O3_____20230304T183806_20230304T184306_27926_03_020401_20230304T191628_2_2_2_2.npy || Path: 1-2-4-8-16 || Timer: 0.2959 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230304T183806_20230304T184306_27926_03_020401_20230304T191628_2_8.npy || Path: 1-2-16 || Timer: 0.0107 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230306T110806_20230306T111306_27950_03_020401_20230306T115630_16.npy || Path: 1-16 || Timer: 0.0061 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230306T110806_20230306T111306_27950_03_020401_20230306T115630_8_2.npy || Path: 1-8-16 || Timer: 0.2885 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230306T110806_20230306T111306_27950_03_020401_20230306T115630_4_4.npy || Path: 1-4-16 || Timer: 0.0109 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230306T110806_20230306T111306_27950_03_020401_20230306T115630_2_4_2.npy || Path: 1-2-8-16 || Timer: 0.3175 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230306T110806_20230306T111306_27950

===> Processing: down_S5P_NRTI_L2__O3_____20230306T125306_20230306T125806_27951_03_020401_20230306T133758_4_2_2.npy || Path: 1-4-8-16 || Timer: 0.2749 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230306T125306_20230306T125806_27951_03_020401_20230306T133758_2_2_4.npy || Path: 1-2-4-16 || Timer: 0.0382 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230306T125306_20230306T125806_27951_03_020401_20230306T133758_2_2_2_2.npy || Path: 1-2-4-8-16 || Timer: 0.2960 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230306T125306_20230306T125806_27951_03_020401_20230306T133758_2_8.npy || Path: 1-2-16 || Timer: 0.0108 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T121806_20230303T122306_27908_03_020401_20230303T130430_16.npy || Path: 1-16 || Timer: 0.0060 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T121806_20230303T122306_27908_03_020401_20230303T130430_8_2.npy || Path: 1-8-16 || Timer: 0.2880 sec.
===> Processing: down_S5P_NRTI_L2__O3_____20230303T121806_20230303T122306_2

===> Processing: down_S5P_RPRO_L2__O3_____20220801T163712_20220801T181841_24875_03_020401_20230123T153306_2_4_2.npy || Path: 1-2-8-16 || Timer: 3.7811 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220801T163712_20220801T181841_24875_03_020401_20230123T153306_4_2_2.npy || Path: 1-4-8-16 || Timer: 3.8244 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220801T163712_20220801T181841_24875_03_020401_20230123T153306_2_2_4.npy || Path: 1-2-4-16 || Timer: 0.1124 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220801T163712_20220801T181841_24875_03_020401_20230123T153306_2_2_2_2.npy || Path: 1-2-4-8-16 || Timer: 4.3789 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220801T163712_20220801T181841_24875_03_020401_20230123T153306_2_8.npy || Path: 1-2-16 || Timer: 0.0106 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220802T175936_20220802T194105_24890_03_020401_20230123T155801_16.npy || Path: 1-16 || Timer: 0.0057 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220802T175936_20220802T1941

===> Processing: down_S5P_RPRO_L2__O3_____20220801T095114_20220801T113243_24871_03_020401_20230123T152043_8_2.npy || Path: 1-8-16 || Timer: 1.1879 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220801T095114_20220801T113243_24871_03_020401_20230123T152043_4_4.npy || Path: 1-4-16 || Timer: 0.0117 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220801T095114_20220801T113243_24871_03_020401_20230123T152043_2_4_2.npy || Path: 1-2-8-16 || Timer: 1.2705 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220801T095114_20220801T113243_24871_03_020401_20230123T152043_4_2_2.npy || Path: 1-4-8-16 || Timer: 1.2996 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220801T095114_20220801T113243_24871_03_020401_20230123T152043_2_2_4.npy || Path: 1-2-4-16 || Timer: 0.0529 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220801T095114_20220801T113243_24871_03_020401_20230123T152043_2_2_2_2.npy || Path: 1-2-4-8-16 || Timer: 1.4669 sec.
===> Processing: down_S5P_RPRO_L2__O3_____20220801T095114_20220801T1